<a href="https://colab.research.google.com/github/girish17work/Data-Analysis/blob/main/Unknown_Dataset_2/Unknown_Dataset_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [25]:
DATASET_URL = 'https://drive.google.com/uc?export=download&id=12Q0OeKkVJbMtnJNH5GrMIr7INvT_4MKt'
df = pd.read_csv(DATASET_URL)
df.head()

,Age,Experience,Projects,HoursPerWeek,EducationYears,Department,EducationLevel,City,Salary
0,41,18.0,24.0,NaN,12,IT,HighSchool,Kolkata,104536
1,23,0.0,12.0,24.0,12,Finance,PhD,Delhi,35804
2,26,0.0,30.0,NaN,25,IT,Bachelors,Bengaluru,34408
3,48,NaN,5.0,NaN,11,HR,Masters,NaN,105748
4,60,26.0,43.0,40.0,22,HR,HighSchool,Bengaluru,123885


In [26]:
print(f"The city with the lowest  percentage of employees: {(df['City'].value_counts(normalize=True)*100).idxmin()}")

The city with the lowest  percentage of employees: Kolkata


In [27]:
q1 = df['Salary'].quantile(0.25)
q3 = df['Salary'].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
higher = q3 + 1.5 * iqr

print(f"The number of outlier in the Salary column: {((df['Salary']<lower) | (df['Salary']>higher)).sum()}")

The number of outlier in the Salary column: 4


In [28]:
print(f"The number of employees in Delhi who works more than 40 hours: {((df['City']=='Delhi') & (df['HoursPerWeek']>40)).sum()}")

The number of employees in Delhi who works more than 40 hours: 36


In [29]:
cv = df.groupby('Department')['Salary'].agg(["mean", "std"])
cv['CV'] = cv['std']/cv['mean']
print(f"The department with the smallest variation in salary, measured using the Coefficient of Variation: {cv['CV'].idxmin()}")

The department with the smallest variation in salary, measured using the Coefficient of Variation: HR


In [30]:
corr = df.corr(numeric_only=True)['Salary'].drop('Salary')
print(f"The feature exhibiting the highest positive correlation with the Salary is {corr.idxmax()} and the value of this correlation is {round(corr.max(), 2)}")

The feature exhibiting the highest positive correlation with the Salary is Experience and the value of this correlation is 0.73


In [31]:
from sklearn.impute import SimpleImputer

imputed_df = df.copy()

numerical_cols = imputed_df.select_dtypes(include="number").columns
categorical_cols = imputed_df.select_dtypes(exclude="number").columns

imputed_df[numerical_cols] = SimpleImputer(strategy='median').fit_transform(imputed_df[numerical_cols])
imputed_df[categorical_cols] = SimpleImputer(strategy='most_frequent').fit_transform(imputed_df[categorical_cols])

print(f"The feature with the lowest standard deviation after imputation is {imputed_df.std(numeric_only=True).idxmin()}")

The feature with the lowest standard deviation after imputation is EducationYears


In [32]:
new_df = df.copy()
new_df['NonEducationYears'] = new_df['Age'] - new_df['EducationYears']
print(f"The number of employees whose ratio NonEducationYears / Age is greater than 0.7 and who earn a salary greater than 100000 is {(((new_df['NonEducationYears']/new_df['Age']) > 0.7) & (new_df['Salary']>100000)).sum()}")

The number of employees whose ratio NonEducationYears / Age is greater than 0.7 and who earn a salary greater than 100000 is 43


In [33]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

encoded_df = imputed_df.copy()

one_hot_encoder = OneHotEncoder(sparse_output=False).set_output(transform='pandas')
city_df = one_hot_encoder.fit_transform(encoded_df[['City']])

ordinal_encoder = OrdinalEncoder(categories=[['HighSchool', 'Bachelors', 'Masters', 'PhD']])
encoded_df['EducationLevel_encoded'] = ordinal_encoder.fit_transform(encoded_df[['EducationLevel']])

encoded_df = pd.concat([encoded_df, city_df], axis=1)
encoded_df.drop(columns=['City', 'EducationLevel', 'Department'])

print(f"Ratio of the mean of the encoded EducationLevel column to the total number of columns: {round(encoded_df['EducationLevel_encoded'].mean()/encoded_df.shape[1], 3)}")

Ratio of the mean of the encoded EducationLevel column to the total number of columns: 0.087


In [34]:
DATASET_URL = 'https://drive.google.com/uc?export=download&id=1c_G-T45RVR3kWTXzeozXPk520sgPyYSp'
df = pd.read_csv(DATASET_URL)
df.head()

,Age,Experience,Projects,HoursPerWeek,EducationYears,Department,EducationLevel,City,Salary
0,49.000000,19.000000,29.421355,46.364626,21.000000,1,1,0,108409.000000
1,60.000000,37.005052,22.449864,52.000000,16.000000,1,0,0,146392.000000
2,23.000000,0.000000,29.000000,45.569474,21.000000,1,0,0,46209.000000
3,23.421598,15.000000,14.100696,76.000000,16.102583,2,3,3,50085.615036
4,58.000000,33.000000,34.000000,38.020632,16.000000,0,0,2,141745.000000


In [35]:
from sklearn.model_selection import train_test_split
X = df.drop(columns=['Salary'])
y = df['Salary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [36]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

estimator = LinearRegression()
rfe = RFE(estimator=estimator, n_features_to_select=0.7)

rfe.fit(X_train, y_train)
print(X_train.columns[rfe.ranking_.argmin()])

Age


In [37]:
from sklearn.linear_model import Lasso
from sklearn.metrics import explained_variance_score

lasso = Lasso(fit_intercept=False, warm_start=True, max_iter=500)
lasso.fit(X_train, y_train)

y_pred = lasso.predict(X_test)
print(f"The explained variance score on the test set is {round(explained_variance_score(y_test, y_pred), 2)}")

The explained variance score on the test set is 0.86


In [38]:
from sklearn.ensemble import VotingRegressor, RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression

voting_regressor = VotingRegressor([
    ('lr', LinearRegression()),
    ('rf', RandomForestRegressor(n_estimators=50, random_state=0)),
    ('knn', KNeighborsRegressor(n_neighbors=4, metric='euclidean'))
])

voting_regressor.fit(X_train, y_train)
print(f"Average of the predicted salary on the test dataset: {round(voting_regressor.predict(X_test).mean(), 2)}")

Average of the predicted salary on the test dataset: 79777.44


In [39]:
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score

pipeline = Pipeline([('pca', PCA()), ('ridge', Ridge())])
param_grid = {
    'pca__n_components': [0.9, 0.95],
    'ridge__alpha': [10, 1, 0.01, 0.001]
}

grid_search_cv = GridSearchCV(pipeline, param_grid=param_grid, cv=5, n_jobs=-1, scoring='neg_mean_absolute_error')
grid_search_cv.fit(X_train, y_train)
best_estimator = grid_search_cv.best_estimator_
y_pred = best_estimator.predict(X_test)
print(f'R2 score on the test dataset: {round(r2_score(y_test, y_pred), 2)}')

R2 score on the test dataset: 0.87


In [40]:
print(f'Proportion of the total variance in the training features explained by the second principal component: {best_estimator.named_steps['pca'].explained_variance_ratio_[1]}')

Proportion of the total variance in the training features explained by the second principal component: 0.35947772875178713


In [41]:
from sklearn.tree import DecisionTreeRegressor

decision_tree_regressor = DecisionTreeRegressor(max_depth=10, min_samples_split=12, random_state=0)
decision_tree_regressor.fit(X_train, y_train)

print(f"The value of the feature importance assigned to the most important feature is {round(decision_tree_regressor.feature_importances_.max(), 2)}")

The value of the feature importance assigned to the most important feature is 0.89


In [42]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import RadiusNeighborsRegressor

X_train_scaled = StandardScaler().fit_transform(X_train)

radii = [0.1, 0.5, 1, 1.5, 2, 2.5, 3, 3.5]

best_radius = None
best_score = float('-inf')

for r in radii:
  estimator = RadiusNeighborsRegressor(radius=r)
  try:
    score = cross_val_score(estimator, X_train_scaled, y_train, cv=5, scoring='r2').mean()
    if score > best_score:
      best_score = score
      best_radius = r
  except ValueError:
    pass

print(f"The best radius is {best_radius}")

The best radius is 3


In [44]:
from sklearn.ensemble import AdaBoostRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error

estimator = AdaBoostRegressor(random_state=0)

param_grid = {
    'n_estimators': [100, 150, 200],
    'learning_rate': [0.0001, 0.001, 0.1]
}

grid_search_cv = GridSearchCV(estimator=estimator, param_grid=param_grid, cv=2)
grid_search_cv.fit(X_train, y_train)

best_estimator = grid_search_cv.best_estimator_
y_pred = best_estimator.predict(X_test)

print(f'Mean Squared Error on the Test Set: {round(mean_squared_error(y_test, y_pred), 2)}')

Mean Squared Error on the Test Set: 126693796.84


In [45]:
from sklearn.tree import DecisionTreeRegressor

decision_tree_regressor = DecisionTreeRegressor(
    criterion='squared_error',
    max_depth=7,
    min_samples_split=3,
    min_impurity_decrease=0.01,
    random_state=0
)

decision_tree_regressor.fit(X_train, y_train)
print(f'The depth of the trained tree: {decision_tree_regressor.get_depth()}')

The depth of the trained tree: 7


In [47]:
from sklearn.tree import DecisionTreeRegressor

decision_tree_regressor = DecisionTreeRegressor(
    criterion='squared_error',
    max_depth=5,
    min_samples_split=3,
    min_impurity_decrease=0.01,
    random_state=0
)

decision_tree_regressor.fit(X_train, y_train)
print(f'Number of nodes in the new tree: {decision_tree_regressor.tree_.node_count}')

Number of nodes in the new tree: 59


In [49]:
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform
from sklearn.metrics import r2_score

estimator = MLPRegressor(random_state=0)

param_distributions = {
    'hidden_layer_sizes': [(50,), (100,), (50,50)],
    'activation': ['tanh', 'relu'],
    'alpha': uniform(loc=1e-5, scale=1e-2)
}

randomized_search_cv = RandomizedSearchCV(
    estimator=estimator,
    param_distributions=param_distributions,
    cv=5,
    scoring='r2',
    random_state=0
)
randomized_search_cv.fit(X_train, y_train)

best_estimator = randomized_search_cv.best_estimator_
y_pred = best_estimator.predict(X_test)

print(f'R2 score on the test set for the best model after tuning: {round(r2_score(y_test, y_pred), 2)}')

R2 score on the test set for the best model after tuning: -7.09
